# Emotional Support Conversations — Data Story

This notebook processes the [ConvoKit emotional-support corpus](https://convokit.cornell.edu/documentation/emotionalsupport.html) (1,300 dyadic support conversations) to extract:

- **Survey score delta** (`final_emotion_intensity − initial_emotion_intensity`) to classify each conversation as `improved`, `flat`, or `declined`
- **VADER sentiment** for each seeker utterance, in turn order

Outputs: `journeys.csv` and `metadata.csv`

## 1. Install dependencies

In [1]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "convokit", "vaderSentiment", "pandas", "numpy",
    "--quiet"
])
print("Dependencies ready.")


[notice] A new release of pip is available: 23.3.1 -> 26.1.1
[notice] To update, run: python3.11 -m pip install --upgrade pip


Dependencies ready.


## 2. Download and load corpus

In [2]:
import convokit
import pandas as pd
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

print("Loading corpus (downloads on first run, cached after)...")
corpus = convokit.Corpus(filename=convokit.download("emotional-support"))

convos = list(corpus.iter_conversations())
print(f"Conversations : {len(convos)}")
print(f"Utterances    : {len(list(corpus.iter_utterances()))}")

Loading corpus (downloads on first run, cached after)...
Dataset already exists at /Users/marissabenz/.convokit/saved-corpora/emotional-support
Conversations : 1300
Utterances    : 38365


## 3. Inspect metadata structure

Before assuming field names, check what keys actually exist.

In [3]:
sample = convos[0]
print("Conversation-level meta keys:", list(sample.meta.keys()))
print("survey_score structure:")
for role, scores in sample.meta["survey_score"].items():
    print(f"  [{role}]: {scores}")

print("\nFirst 5 utterances:")
for i, utt in enumerate(sample.iter_utterances()):
    print(f"  speaker={utt.speaker.id:<20} text={utt.text[:60]!r}")
    if i >= 4:
        break

Conversation-level meta keys: ['experience_type', 'emotion_type', 'problem_type', 'situation', 'survey_score', 'seeker_question1', 'seeker_question2', 'supporter_question1', 'supporter_question2']
survey_score structure:
  [seeker]: {'initial_emotion_intensity': '5', 'empathy': '5', 'relevance': '5', 'final_emotion_intensity': '1'}
  [supporter]: {'relevance': '5'}

First 5 utterances:
  speaker=seeker_0             text='Hello\n'
  speaker=supporter_0          text='Hello, what would you like to talk about?'
  speaker=seeker_0             text='I am having a lot of anxiety about quitting my current job. '
  speaker=supporter_0          text='What makes your job stressful for you?'
  speaker=seeker_0             text='I have to deal with many people in hard financial situations'


## 4. Process all conversations

For each conversation:
- Extract `delta = final_emotion_intensity − initial_emotion_intensity` (both are seeker self-reports, 1–5 scale)
- Classify: `much improved` (delta ≤ -2), `somewhat improved` (delta = -1), `little change` (delta ≥ 0), `unknown` (score missing)
- Run VADER on every seeker utterance in turn order

In [7]:
analyzer = SentimentIntensityAnalyzer()

journey_rows = []
meta_rows    = []
missing_final_score = 0

for convo in corpus.iter_conversations():
    cid  = convo.id
    meta = convo.meta

    # --- Survey score delta ---
    try:
        seeker_scores = meta["survey_score"]["seeker"]
        score1 = float(seeker_scores["initial_emotion_intensity"])
        score2 = float(seeker_scores["final_emotion_intensity"])
        delta  = score2 - score1
    except (KeyError, TypeError, ValueError):
        score1 = score2 = delta = None
        missing_final_score += 1

    # delta is negative = improved (lower intensity = feeling better)
    # Spread into three groups based on how much they improved
    if pd.isna(delta):
        cluster = "unknown"
    elif delta <= -2:
        cluster = "much improved"
    elif delta == -1:
        cluster = "somewhat improved"
    else:
        cluster = "little change"

    # --- VADER on seeker utterances ---
    turn = 0
    for utt in convo.iter_utterances():
        if "seeker" in utt.speaker.id:
            compound = analyzer.polarity_scores(utt.text or "")["compound"]
            journey_rows.append({
                "conversation_id": cid,
                "turn":            turn,
                "sentiment":       compound,
                "cluster":         cluster,
            })
            turn += 1

    # --- Metadata row ---
    meta_rows.append({
        "conversation_id": cid,
        "cluster":         cluster,
        "emotion_type":    meta.get("emotion_type", ""),
        "problem_type":    meta.get("problem_type", ""),
        "delta":           delta,
    })

journeys_df = pd.DataFrame(journey_rows)
metadata_df = pd.DataFrame(meta_rows)

print(f"Journey rows : {len(journeys_df):,}")
print(f"Metadata rows: {len(metadata_df):,}")
print(f"Conversations missing final score: {missing_final_score} "
      f"(no final_emotion_intensity recorded — labeled 'unknown')")

Journey rows : 19,989
Metadata rows: 1,300
Conversations missing final score: 150 (no final_emotion_intensity recorded — labeled 'unknown')


## 5. Cluster counts

In [8]:
counts = metadata_df["cluster"].value_counts()
print("Conversations per cluster:")
print(counts.to_string())
print(f"\nTotal: {counts.sum()}")

print("\nDelta summary for scored conversations:")
print(metadata_df.dropna(subset=["delta"])["delta"].describe().round(2))

Conversations per cluster:
cluster
much improved        733
somewhat improved    417
unknown              150

Total: 1300

Delta summary for scored conversations:
count    1150.00
mean       -1.99
std         0.95
min        -4.00
25%        -3.00
50%        -2.00
75%        -1.00
max        -1.00
Name: delta, dtype: float64


## 6. Save CSV files

In [9]:
import pathlib

out_dir = pathlib.Path().resolve()   # same folder as this notebook

journeys_path = out_dir / "journeys.csv"
metadata_path = out_dir / "metadata.csv"

journeys_df.to_csv(journeys_path, index=False)
metadata_df.to_csv(metadata_path, index=False)

print(f"Saved: {journeys_path}")
print(f"Saved: {metadata_path}")

print("\njourneys.csv — first 5 rows:")
display(journeys_df.head())

print("metadata.csv — first 5 rows:")
display(metadata_df.head())

Saved: /Users/marissabenz/Dartmouth College Dropbox/Marissa Benz/marissa/journeys.csv
Saved: /Users/marissabenz/Dartmouth College Dropbox/Marissa Benz/marissa/metadata.csv

journeys.csv — first 5 rows:


,conversation_id,turn,sentiment,cluster
0,conversation_0,0,0.0000,much improved
1,conversation_0,1,0.0387,much improved
2,conversation_0,2,-0.5423,much improved
3,conversation_0,3,0.0000,much improved
4,conversation_0,4,0.6310,much improved


metadata.csv — first 5 rows:


,conversation_id,cluster,emotion_type,problem_type,delta
0,conversation_0,much improved,anxiety,job crisis,-4.0
1,conversation_1,much improved,anger,problems with friends,-4.0
2,conversation_2,much improved,fear,job crisis,-2.0
3,conversation_3,somewhat improved,depression,ongoing depression,-1.0
4,conversation_4,somewhat improved,depression,breakup with partner,-1.0
